# Equity Intelligence Platform - demo

This notebook shows how to *use* the platform. The heavy lifting lives in the `equity_intel` package; run the full pipeline first with `python -m equity_intel run`.

The original research notebook is kept in `notebooks/archive/`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
from equity_intel.config import load_config
from equity_intel.data import warehouse as wh
pd.set_option('display.width', 160)
cfg = load_config()

## 1. What is in the warehouse?

In [2]:
wh.list_tables(cfg)

,table_name,table_type
0,dim_stock,BASE TABLE
1,dq_report,BASE TABLE
2,drift_report,BASE TABLE
3,fact_benchmark,BASE TABLE
4,fact_daily_returns,BASE TABLE
5,fact_portfolio_weights,BASE TABLE
6,fact_predictions,BASE TABLE
7,fact_prices,BASE TABLE
8,fact_trades,BASE TABLE
9,fact_transactions,BASE TABLE


## 2. Backtest results vs NIFTY 50

In [3]:
wh.query(cfg, '''SELECT period, Start, "End", round(Total_Return, 2) AS strategy_pct,
       round(Benchmark_Return, 2) AS nifty_pct, round(Sharpe, 2) AS sharpe, round(MaxDD, 2) AS max_dd_pct
FROM perf_summary ORDER BY Start''')

,period,Start,End,strategy_pct,nifty_pct,sharpe,max_dd_pct
0,Validation,2024-01-01,2024-06-28,13.71,10.43,1.88,-6.77
1,Testing,2024-07-01,2024-12-31,-5.10,-2.06,-0.39,-15.08
2,Holdout_2025,2025-01-01,2025-12-31,10.33,10.05,1.26,-7.73
3,Holdout_2026_YTD,2026-01-01,2026-11-23,-7.72,-3.05,-0.21,-15.76


## 3. Is the model better than simple alternatives?

Same costs and portfolio rules for every strategy. `Random_Rank` shows what the rules achieve with no predictive skill at all.

In [4]:
comp = wh.query(cfg, 'SELECT * FROM strategy_comparison')
comp.pivot_table(index='strategy', columns='period', values='total_return', sort=False).round(2)

period,Validation,Testing,Holdout_2025,Holdout_2026_YTD
strategy,,,,
ML_Ensemble (static),13.71,-5.10,10.33,-7.72
NIFTY_50,10.43,-2.06,10.05,-3.05
ML_Ensemble (walk-forward),15.11,-6.20,7.40,-7.35
Equal_Weight,13.85,-3.79,10.22,-0.96
Momentum_Only,15.48,-10.44,-0.31,1.18
Random_Rank (mean of 10),13.93,-6.76,4.00,-6.06


## 4. Today's model ranking and why

In [5]:
wh.query(cfg, 'SELECT rank, symbol, sector, round(score * 100, 2) AS predicted_5d_pct '
              'FROM v_latest_signals LIMIT 10')

,rank,symbol,sector,predicted_5d_pct
0,1,BAJFINANCE.NS,NBFC,1.16
1,2,BAJAJ-AUTO.NS,Auto,1.01
2,3,HDFCBANK.NS,Banking,0.94
3,4,SBIN.NS,Banking,0.91
4,5,LTIM.NS,IT,0.85
5,6,LT.NS,Infrastructure,0.81
6,7,APOLLOHOSP.NS,Healthcare,0.75
7,8,TECHM.NS,IT,0.72
8,9,CIPLA.NS,Pharma,0.67
9,10,WIPRO.NS,IT,0.66


In [6]:
wh.query(cfg, 'SELECT symbol, top_drivers FROM latest_explanations LIMIT 5')

,symbol,top_drivers
0,BAJFINANCE.NS,"bb_width (+0.31%), bb_lower (+0.17%), vol60 (-..."
1,BAJAJ-AUTO.NS,"vol60 (+0.18%), vol_chg5 (+0.14%), vol20 (+0.13%)"
2,HDFCBANK.NS,"bb_width (+0.22%), macd_hist (+0.16%), close_o..."
3,SBIN.NS,"bb_width (+0.21%), macd_hist (+0.20%), atr_z (..."
4,LTIM.NS,"atr14 (-0.17%), vol_chg5 (+0.15%), vol_sma_rat..."


## 5. Current portfolio risk

In [7]:
wh.query(cfg, 'SELECT * FROM portfolio_risk').T

,0
as_of,2026-11-23 00:00:00
n_holdings,15
portfolio_volatility,0.142332
portfolio_beta,0.893288
VaR_95_1d,0.015663
CVaR_95_1d,0.020432
diversification_ratio,1.699768
effective_n_stocks,13.660955
cash_weight,0.0
VaR_95_1d_inr,15662.910052


In [8]:
wh.query(cfg, 'SELECT scenario, type, round(portfolio_return_pct, 2) AS portfolio_pct, '
              'round(nifty_return_pct, 2) AS nifty_pct FROM stress_tests')

,scenario,type,portfolio_pct,nifty_pct
0,Demonetisation_2016,historical,-5.77,-7.44
1,COVID_crash_2020,historical,-33.44,-37.24
2,Rate_hikes_2022,historical,-8.30,-16.47
3,Adani_Hindenburg_2023,historical,-6.20,-4.49
4,Correction_H2_2024,historical,-14.60,-10.81
5,Market_down_10pct,hypothetical,-8.93,NaN
6,Market_down_20pct,hypothetical,-17.87,NaN
7,Banking_down_15pct,hypothetical,-2.55,NaN
8,IT_down_15pct,hypothetical,-0.00,NaN


## 6. Ask a question in plain English

Uses Claude when `ANTHROPIC_API_KEY` is set (or a local Ollama model); otherwise it falls back to the closest pre-written business query.

In [9]:
from equity_intel.genai.text_to_sql import ask
a = ask(cfg, 'Which sectors does the model buy most often?')
print(a.provider)
print(a.answer)
a.result.head(10)

offline (no language model)
Offline mode (no language model configured): showing the closest pre-written business query - **Q3. Which sectors does the model favour? (share of all BUY orders)**. Configure a model to get free-form answers.


,sector,buys,pct_of_buys
0,Consumer,136,15.8
1,Auto,107,12.4
2,IT,105,12.2
3,Banking,99,11.5
4,Energy,86,10.0
5,Pharma,71,8.2
6,NBFC,54,6.3
7,Metals,52,6.0
8,Infrastructure,41,4.8
9,Insurance,33,3.8
